In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 84.9 MB/s eta 0:00:00


In [ ]:
import faiss
import string
import numpy as np
import pandas as pd

from collections import Counter
from transformers import AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer, CrossEncoder

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer


train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

print(train.info())
print("Creating Knowledge Base")

kb = []

for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

# Loading the embedding model and creating the FAISS index
print("Loading embedding model and creating index")

model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=False)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB
None
Creating knowledge base
Loading embedding model and creating index
Knowledge base successfully created


In [ ]:
# Loading models for zero-shot classification, reranking, and tokenization
zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Extracting row 150 data for Q1, Q2, Q5, Q6
row_150 = train.iloc[150]

prompt_150 = str(row_150['prompt'])
options = ["A", "B", "C", "D", "E"]

labels_150 = [str(row_150[opt]) for opt in options]
ans_150 = str(row_150[row_150['answer']])

# Zero-shot classifier for Q1, Q2, Q6

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)

In [4]:
result_q1 = zs(prompt_150, labels_150)

for label, score in zip(result_q1['labels'], result_q1['scores']):
    if label == ans_150:
        answer_q1 = round(score, 3)
        print(f"Q1 answer: {answer_q1}")
        break

Q1 answer: 0.384


Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [5]:
prompt_emb = model.encode([prompt_150], show_progress_bar=False)
prompt_emb = np.array(prompt_emb, dtype=np.float32)

k = 10
distances, indices = index.search(prompt_emb, k)
retrieved_indices = indices[0].tolist()

if 150 in retrieved_indices:
    rank = retrieved_indices.index(150) + 1
    print(f"Q2 answer: rank {rank}")
else:
    print("True document NOT found in top 10")

Q2 answer: rank 10


Concept: The Two-Stage Pipeline (Reranking & Cross-Encoders)

In Question 2, you saw that our FAISS database did not put the true document at rank #1. Why? Because FAISS uses Bi-encoder. A Bi-encoder embeds the question and the document separately and just compares the distance (Cosine Similarity). They are fast and allows you to search millions of documents but often miss semantic context.

To fix this we use a 2 stage pipeline:
1. Retrieval: Use a bi-encoder with FAISS to quickly get the top k possible chunks/documents.
2. Reranking: Use a Cross-Encoder to deeply evaluate those top candidates and sort them based on highest semantic similarity.

A Cross-Encoder passes the Question and the Document into the Transformer network at the exact same time. The Attention mechanism can directly compare the words in the question to the words in the document, resulting in a highly accurate relevance score.

For this we pass the prompt and the context and ask it to predict a score.

Cross Encoders (Hugging Face) - https://huggingface.co/cross-encoder

# Code to use a Cross-Encoder

In [ ]:
# Getting the top 10 documents from FAISS and scoring them with the Cross-Encoder
docs_10 = [kb[i] for i in retrieved_indices]
pairs = [[prompt_150, doc] for doc in docs_10]

ce_scores = cross_encoder.predict(pairs)

Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [7]:
ranked_by_ce = sorted(zip(retrieved_indices, ce_scores), key=lambda x: x[1], reverse=True)

for rank, (orig_idx, score) in enumerate(ranked_by_ce, 1):
    if orig_idx == 150:
        print(f"Q3 answer: rank {rank}")
        break

Q3 answer: rank 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?

In [8]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

prompt_42_emb = model.encode([prompt_42], show_progress_bar=False)
prompt_42_emb = np.array(prompt_42_emb, dtype=np.float32)

_, indices_42 = index.search(prompt_42_emb, 5)
retrieved_42 = indices_42[0].tolist()

docs_5 = [kb[i] for i in retrieved_42]

concatenated = " ".join(docs_5)

rag_string_42 = f"Context: {concatenated} Question: {prompt_42}"

tokens = tokenizer(rag_string_42, truncation=False)
answer_q4 = len(tokens['input_ids'])
print(f"Q4 answer: {answer_q4}")

Q4 answer: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).

In [9]:
true_doc = kb[150]

rag_string_150 = f"Context: {true_doc} Question: {prompt_150}"

result_q5 = zs(rag_string_150, labels_150)

for label, score in zip(result_q5['labels'], result_q5['scores']):
    if label == ans_150:
        answer_q5 = round(score, 3)
        print(f"Q5 answer: {answer_q5}")
        break

Q5 answer: 0.989


Concept: The Danger of Bad Retrieval (Adversarial RAG)

The golden rule of Retrieval-Augmented Generation is "Garbage In, Garbage Out." An LLM places immense trust in the external context you inject into its prompt. If your vector database performs poorly and retrieves an irrelevant or incorrect document, the model will often abandon its own internal reasoning and confidently generate the wrong answer based on that bad data. We do the reranking and constricting the number of chunks that we give to the model for the same reason.

Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [10]:
wrong_doc = kb[999]
adversarial_rag = f"Context: {wrong_doc} Question: {prompt_150}"

result_q6 = zs(adversarial_rag, labels_150)

for label, score in zip(result_q6['labels'], result_q6['scores']):
    if label == ans_150:
        answer_q6 = round(score, 3)
        print(f"Q6 answer: {answer_q6}")
        break

Q6 answer: 0.529


In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).

In [11]:
hits = 0

for i in range(100):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    correct_doc = str(row[row['answer']])

    prompt_emb_i = model.encode([prompt], show_progress_bar=False)
    prompt_emb_i = np.array(prompt_emb_i, dtype=np.float32)

    _, idxs = index.search(prompt_emb_i, 5)
    retrieved_docs = [kb[j] for j in idxs[0]]

    if any(correct_doc in doc for doc in retrieved_docs):
        hits += 1

hit_rate = round(hits / 100 * 100, 1)
print(f"Q7 answer: {hit_rate}%")

Q7 answer: 73.0%


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [ ]:
# Mean Average Precision @ 3
def map3_score(actual, predictions):
    for i, prediction in enumerate(predictions, start=1):
        if prediction == actual:
            return 1 / i
    return 0


options = ["A", "B", "C", "D", "E"]

scores = []

for i in range(20):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    labels = [str(row[opt]) for opt in options]

    p_emb = model.encode([prompt], show_progress_bar=False)
    p_emb = np.array(p_emb, dtype=np.float32)
    _, idxs = index.search(p_emb, 5)
    docs_5 = [kb[j] for j in idxs[0]]

    pairs_i = [[prompt, doc] for doc in docs_5]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc = docs_5[np.argmax(ce_scores_i)]

    rag_str = f"Context: {best_doc} Question: {prompt}"

    result = zs(rag_str, labels)

    label_to_option = {str(row[opt]): opt for opt in options}
    ranked_labels = result['labels']
    ranked_options = []
    
    for lbl in ranked_labels:
        if lbl in label_to_option:
            ranked_options.append(label_to_option[lbl])

    scores.append(
        map3_score(row['answer'], ranked_options[:3])
    )

final_map3 = round(sum(scores) / len(scores), 3)
print(f"Q8 answer: {final_map3}")

Q8 answer: 0.975
